## Preparar el entorno

In [ ]:
%pip install pandas pyarrow plotly ipython nbformat

## Recuperar los datos incorporados en el notebook

In [3]:
import base64
from pathlib import Path
import tempfile
import pandas as pd
import plotly.express as px
import nbformat

DATOS_B64 = '''...'''

parquet_path = Path(tempfile.gettempdir()) / 'sismos_peru_gold.parquet'
parquet_path.write_bytes(base64.b64decode(DATOS_B64))
df = pd.read_parquet(parquet_path)
df['semana'] = df['fecha_hora_utc'].dt.isocalendar().week.astype(int)
print(f'nbformat: {nbformat.__version__}')
print(f'{len(df):,} sismos cargados')
print(f"Periodo: {df['fecha_hora_utc'].min():%Y-%m-%d} a {df['fecha_hora_utc'].max():%Y-%m-%d}")
df.head()

nbformat: 5.11.0
3,444 sismos cargados
Periodo: 2000-01-06 a 2026-08-11


,id,fecha_hora_utc,magnitud,mag_type,lugar,profundidad_km,latitud,longitud,alerta,tsunami,...,tipo_epicentro,distancia_costa_km,anio,mes,hora,magnitud_categoria,profundidad_categoria,dias_desde_ultimo_sismo_mismo_departamento,magnitud_zscore_departamento,semana
0,usp0009t15,2000-05-15 22:58:04.980000+00:00,4.5,mb,"18 km WNW of Pujilí, Ecuador",33.000000,-0.910,-78.859001,NaN,0,...,Mar,258.5,2000,5,22,Leve,Superficial,NaN,-0.905441,20
1,usp0009xz6,2000-08-05 19:25:59.110000+00:00,4.6,mb,"53 km NNE of Yambrasbamba, Peru",33.000000,-5.266,-77.799004,NaN,0,...,Punto fijo,NaN,2000,8,19,Moderado,Superficial,81.0,-0.596769,31
2,usp0009xzs,2000-08-06 05:15:41.320000+00:00,4.9,mb,"54 km N of Yambrasbamba, Peru",33.000000,-5.249,-77.832001,NaN,0,...,Punto fijo,NaN,2000,8,5,Moderado,Superficial,0.0,0.329251,31
3,usp0009ybz,2000-08-13 04:19:02.090000+00:00,5.1,mb,"64 km NNE of Yambrasbamba, Peru",33.000000,-5.174,-77.760002,NaN,0,...,Punto fijo,NaN,2000,8,4,Moderado,Superficial,6.0,0.946596,32
4,usp000a1cg,2000-10-07 00:00:05.440000+00:00,4.6,mb,"30 km NNE of Gualaquiza, Ecuador",94.099998,-3.166,-78.433998,NaN,0,...,Mar,25.9,2000,10,0,Moderado,Intermedio,54.0,-0.596769,40


## Indicadores generales

In [4]:
resumen = pd.DataFrame({
    'Indicador': ['Total de sismos', 'Magnitud promedio', 'Magnitud máxima', 'Profundidad promedio (km)'],
    'Valor': [len(df), round(df['magnitud'].mean(), 2), df['magnitud'].max(), round(df['profundidad_km'].mean(), 1)]
})
resumen

,Indicador,Valor
0,Total de sismos,3444.00
1,Magnitud promedio,4.85
2,Magnitud máxima,8.40
3,Profundidad promedio (km),76.00


## Pregunta 1: ¿Qué departamentos concentran la mayor actividad sísmica?

In [5]:
ranking = df['departamento'].astype(str).value_counts().head(15).sort_values()
fig = px.bar(ranking, orientation='h', text_auto=True,
             labels={'value': 'Cantidad de sismos', 'departamento': 'Departamento'},
             title='Departamentos con mayor actividad sísmica')
fig.update_layout(showlegend=False)
fig.show()

## Pregunta 2: ¿Cuáles fueron los sismos de mayor magnitud?

In [9]:
mayores = df.nlargest(20, 'magnitud').sort_values('magnitud')
mayores[['fecha_hora_utc', 'departamento', 'magnitud', 'lugar']].sort_values('magnitud', ascending=False)

,fecha_hora_utc,departamento,magnitud,lugar
226,2001-06-23 20:33:14.130000+00:00,Arequipa,8.4,"6 km SSW of Atico, Peru"
1186,2007-08-15 23:40:57.890000+00:00,Ica,8.0,"41 km SW of San Vicente de Cañete, Peru"
2046,2019-05-26 07:41:15.073000+00:00,Loreto,8.0,"78 km NE of Navarro, Peru"
366,2001-07-07 09:38:43.520000+00:00,Arequipa,7.6,"51 km SW of Punta de Bombón, Peru"
2187,2015-11-24 22:45:38.880000+00:00,Madre de Dios,7.6,"155 km WNW of Iñapari, Peru"
3347,2015-11-24 22:50:54.370000+00:00,Ucayali,7.6,"185 km WNW of Iñapari, Peru"
1840,2005-09-26 01:55:37.670000+00:00,Loreto,7.5,"39 km NW of Yurimaguas, Peru"
2037,2019-02-22 10:17:23.770000+00:00,Loreto,7.5,"115 km ESE of Palora, Ecuador"
2093,2021-11-28 10:52:14.579000+00:00,Loreto,7.5,"43 km NNW of Barranca, Peru"
871,2024-06-28 05:36:36.902000+00:00,Arequipa,7.2,"10 km WSW of Atiquipa, Peru"


## Pregunta 3: ¿Cómo evolucionaron la frecuencia y la magnitud a través del tiempo?

In [7]:
evolucion = (df.groupby('anio', observed=True)
             .agg(cantidad=('id', 'count'), magnitud_promedio=('magnitud', 'mean'))
             .reset_index())
fig = px.line(evolucion, x='anio', y='cantidad', markers=True, text='cantidad',
              title='Cantidad de sismos por año')
fig.show()
evolucion.tail(27)

,anio,cantidad,magnitud_promedio
0,2000,71,4.866197
1,2001,295,4.938644
2,2002,101,4.844554
3,2003,72,4.929167
4,2004,74,4.872973
5,2005,149,4.936242
6,2006,92,4.863043
7,2007,207,4.936715
8,2008,95,4.800000
9,2009,90,4.928889


## Pregunta 4: ¿Dónde se concentran geográficamente los eventos?

In [12]:
df2 = df[df['tipo_epicentro'] == 'Punto fijo']
fig = px.scatter_map(df2, lat='latitud', lon='longitud', size='magnitud',
                     color='magnitud_categoria', zoom=4, height=650,
                     hover_data=['departamento', 'fecha_hora_utc', 'magnitud'],
                     title='Distribución geográfica de los sismos')
fig.update_layout(map_style='open-street-map')
fig.show()

## Pregunta 5: ¿Qué eventos se alejan del patrón esperado de magnitud y frecuencia?

In [13]:
umbral = 3.0

df["tipo"] = df["magnitud_zscore_departamento"].abs().ge(umbral).map({
    True: "Atípico",
    False: "Típico"
})

df["etiqueta"] = df["departamento"].astype(str).where(
    df["tipo"] == "Atípico", ""
)
atipicos = df[df["tipo"] == "Atípico"].copy()

fig = px.scatter(
    df,
    x="fecha_hora_utc",
    y="magnitud_zscore_departamento",
    color="tipo",
    text="etiqueta",
    hover_data=["departamento", "magnitud", "lugar"],
    title=f"Sismos atípicos por departamento (|z-score| ≥ {umbral})"
)

fig.update_traces(textposition="top center")
fig.add_hline(y=umbral, line_dash="dot")
fig.add_hline(y=-umbral, line_dash="dot")
fig.show()

atipicos.nlargest(
    20, "magnitud_zscore_departamento"
)[[
    "fecha_hora_utc",
    "departamento",
    "magnitud",
    "magnitud_zscore_departamento"
]]

,fecha_hora_utc,departamento,magnitud,magnitud_zscore_departamento
226,2001-06-23 20:33:14.130000+00:00,Arequipa,8.4,8.348668
1186,2007-08-15 23:40:57.890000+00:00,Ica,8.0,7.135618
2046,2019-05-26 07:41:15.073000+00:00,Loreto,8.0,6.563086
366,2001-07-07 09:38:43.520000+00:00,Arequipa,7.6,6.448699
1840,2005-09-26 01:55:37.670000+00:00,Loreto,7.5,5.526894
2037,2019-02-22 10:17:23.770000+00:00,Loreto,7.5,5.526894
2093,2021-11-28 10:52:14.579000+00:00,Loreto,7.5,5.526894
871,2024-06-28 05:36:36.902000+00:00,Arequipa,7.2,5.498714
614,2013-09-25 16:42:43.170000+00:00,Arequipa,7.1,5.261219
718,2018-01-14 09:18:45.540000+00:00,Arequipa,7.1,5.261219
